# Classic Component Making (B-3046295)
- Non-modeled components: expense, cat loads, cost of capital, NCOR to AAP, balance
- Retention training: owned by Yinan, not run here

**Before running**
1. Attach the classic environment (Resources must hold `classic_specialty_ltv-10.0.0/` and `ltv_helpers-2.0.0/`)
2. Set `CLV_lakehouse_classic_spl_STG` as the default lakehouse
3. Restart the session after any change to Resources, then run top to bottom

In [ ]:
RUN_TYPE = "prod"
GIT_CHECKOUT = "feature/B-3046295"  # must match the git_checkout label used by the copy pipeline
MODEL_VERSION = "2.0.0"             # model store version (banner shows model store: v2.0)
PKG_VERSION = "10.0.0"
HELPERS_VERSION = "2.0.0"
EXPECTED_LAKEHOUSE = "CLV_lakehouse_classic_spl_STG"
EXPENSE_MODULE = None               # None = auto-detect; set a dotted path only if auto-detect finds several

## Setup

In [ ]:
import sys

import notebookutils

# Resources hold unpacked sdists. The folder uses the PEP 625 distribution name
# (classic_specialty_ltv-<ver>); the import name inside it is classic_spl_ltv.
SPL_ROOT = f"{notebookutils.nbResPath}/env/classic_specialty_ltv-{PKG_VERSION}"
HELPER_ROOT = f"{notebookutils.nbResPath}/env/ltv_helpers-{HELPERS_VERSION}"

for root in (SPL_ROOT, HELPER_ROOT):
    if root not in sys.path:
        sys.path.insert(0, root)

In [ ]:
import importlib
import os

lakehouse = notebookutils.runtime.context.get("defaultLakehouseName")
assert lakehouse == EXPECTED_LAKEHOUSE, f"Default lakehouse is {lakehouse!r}; attach {EXPECTED_LAKEHOUSE}"

print("env Resources:", sorted(os.listdir(f"{notebookutils.nbResPath}/env")))
for root in (SPL_ROOT, HELPER_ROOT):
    assert os.path.isdir(root), f"Missing in environment Resources: {root}"

# A ModuleNotFoundError here names the package missing from the environment YAML
for name in ("loguru", "ltv_helpers.fabric_pipeline_helpers", "classic_spl_ltv.config.run_paths"):
    importlib.import_module(name)

import classic_spl_ltv
import ltv_helpers

for mod, root in ((classic_spl_ltv, SPL_ROOT), (ltv_helpers, HELPER_ROOT)):
    print(f"{mod.__name__:16s} {mod.__file__}")
    assert mod.__file__.startswith(root), f"{mod.__name__} loaded from outside Resources; restart the session"

## Run paths

In [ ]:
from delta.tables import DeltaTable

from classic_spl_ltv.config import run_paths

p = run_paths.make_paths(RUN_TYPE, GIT_CHECKOUT, MODEL_VERSION)

BASE = f"Files/data/classic_spl_ltv/{RUN_TYPE}/{GIT_CHECKOUT}"  # must equal "base" in the banner above
print("eltv_pipeline inputs present:", notebookutils.fs.exists(f"{BASE}/eltv_pipeline"))


def find_delta_tables(root, depth=4):
    """Folders under root that hold a Delta table (skips the copied inputs)."""
    if depth < 0:
        return []
    subdirs = [f for f in notebookutils.fs.ls(root) if f.isDir]
    if any(f.name == "_delta_log" for f in subdirs):
        return [root]
    out = []
    for f in subdirs:
        if f.name != "eltv_pipeline":
            out += find_delta_tables(f.path, depth - 1)
    return out


def check_delta(path):
    """Fail if no Delta table at path; show last write time and shape."""
    if not notebookutils.fs.exists(f"{path}/_delta_log"):
        print("Delta tables under BASE:", *find_delta_tables(BASE), sep="\n  ")
        raise AssertionError(f"No Delta table at {path}; use the path from the log line")
    DeltaTable.forPath(spark, path).history(1).select("timestamp", "operation").show(truncate=False)
    df = spark.read.format("delta").load(path)
    print(f"{path}\nrows={df.count():,}  cols={len(df.columns)}")

## Non-modeled components
### Expense
Reads the Finance Excel file, so it does not depend on the parquet to Delta conversion.

The old import (`...expense.process_expense_data.process_raw_expense`) came from the renters notebook and does not exist in classic. This cell finds the classic expense job from the package itself.

In [ ]:
import ast
import inspect
from pathlib import Path

pkg_dir = Path(classic_spl_ltv.__file__).parent
exp_dir = pkg_dir / "jobs" / "expense"
assert exp_dir.is_dir(), f"{exp_dir} missing from Resources; rebuild from Yinan's latest commit"

print("jobs/expense layout:")
for py in sorted(exp_dir.rglob("*.py")):
    print("  ", py.relative_to(exp_dir))


def defines_main(py):
    tree = ast.parse(py.read_text(encoding="utf-8"))
    return any(isinstance(n, ast.FunctionDef) and n.name == "main" for n in tree.body)


candidates = sorted(
    ".".join(py.relative_to(pkg_dir.parent).with_suffix("").parts)
    for py in exp_dir.rglob("*.py")
    if py.name != "__init__.py" and defines_main(py)
)
print("modules with main():", candidates)

if EXPENSE_MODULE:
    expense_module = EXPENSE_MODULE
else:
    assert candidates, "No expense job with main() in Resources; send this output to Yinan"
    assert len(candidates) == 1, f"Several candidates; set EXPENSE_MODULE to one of {candidates}"
    expense_module = candidates[0]

prexp = importlib.import_module(expense_module)
print(f"\nusing {expense_module}.main{inspect.signature(prexp.main)}")

In [ ]:
prexp.main(spark)

In [ ]:
# Expected path follows the renters layout; if it differs, check_delta lists what was written
check_delta(f"{BASE}/expense/processed_expense_df")

### Cat loads, cost of capital, NCOR to AAP, balance
Copy these cells from Yinan's `classic_spl_component_making`, one section per component:
1. The import + `main(spark)` cell, unchanged
2. A check cell: `check_delta(f"{BASE}/<path from the log line>")`

Components that read `eltv_pipeline` inputs may fail until Yinan's parquet to Delta conversion lands. Stop at the first error.

## Retention training
Owned by Yinan. Not run in this notebook.